# 02 — Test du mode Documents de Google Traduction

> **Résultat du pilote : voie abandonnée.** Le 22 septembre 2026, Google Traduction a affiché « La traduction de documents n'est pas disponible en Baoulé ». Les fichiers pilotes sont conservés comme trace de l'expérience, mais aucun lot complet ne doit être généré.

L'API Cloud Translation ne répertorie actuellement pas le baoulé (`bau`). Bien que la langue soit disponible dans le mode **Texte**, elle n'est pas disponible dans le mode **Documents**. Ce notebook documente le pilote qui a permis de vérifier cette limitation sans produire inutilement tous les lots.

Le service accepte les classeurs `.xlsx` jusqu'à 10 Mo. Nous commençons par deux pilotes de 100 exemples :

- `fr_bau` : seule la réponse française est envoyée à la traduction ;
- `bau_bau` : instruction, contexte et réponse sont envoyés à la traduction.

La génération complète est maintenant verrouillée. Une autre méthode de traduction doit être validée avant de poursuivre.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import re
import zipfile
from xml.etree import ElementTree as ET

PROJECT_ROOT = Path.cwd()
SPLIT_DIR = PROJECT_ROOT / "data" / "01_splits"
STEP_DIR = PROJECT_ROOT / "data" / "02_google_translate"
PILOT_UPLOAD_DIR = STEP_DIR / "pilot_to_upload"
PILOT_TRANSLATED_DIR = STEP_DIR / "pilot_translated"
FULL_UPLOAD_DIR = STEP_DIR / "full_to_upload"
FULL_TRANSLATED_DIR = STEP_DIR / "full_translated"

PILOT_ROWS = 100
BATCH_SIZE = 2_000
DOCUMENT_TRANSLATION_SUPPORTS_BAOULE = False  # Vérifié dans l'interface Google le 2026-09-22.
GENERATE_FULL_BATCHES = False  # Verrouillé : le mode Documents ne prend pas en charge le baoulé.
MAX_GOOGLE_DOCUMENT_BYTES = 10 * 1024 * 1024
MAX_EXCEL_CELL_CHARS = 32_767

for directory in (PILOT_UPLOAD_DIR, PILOT_TRANSLATED_DIR, FULL_UPLOAD_DIR, FULL_TRANSLATED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Répertoire de l'étape 2 :", STEP_DIR)
if GENERATE_FULL_BATCHES and not DOCUMENT_TRANSLATION_SUPPORTS_BAOULE:
    raise RuntimeError("Génération interdite : Google Documents ne prend pas en charge le baoulé.")
print("Génération complète activée :", GENERATE_FULL_BATCHES)

## 1. Charger et contrôler les deux partitions à traduire

In [ ]:
def read_jsonl(path):
    records = []
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"JSON invalide dans {path}, ligne {line_number}") from exc
    return records

source_partitions = {
    "fr_bau": read_jsonl(SPLIT_DIR / "fr_bau_source.jsonl"),
    "bau_bau": read_jsonl(SPLIT_DIR / "bau_bau_source.jsonl"),
}

required_fields = {"id", "source_row_index", "direction", "instruction_fr", "input_fr", "output_fr"}
for direction, records in source_partitions.items():
    assert records, f"Partition vide : {direction}"
    assert all(required_fields <= set(record) for record in records)
    assert all(record["direction"] == direction for record in records)
    assert len({record["id"] for record in records}) == len(records)
    assert len({record["source_row_index"] for record in records}) == len(records)
    print(f"{direction:8s}: {len(records):,} exemples")

## 2. Écrire des classeurs XLSX sans dépendance externe

La première colonne contient uniquement l'index numérique immuable de la ligne source. Le prochain notebook s'en servira pour rattacher chaque traduction à son exemple original, même si Google traduit les en-têtes.

In [ ]:
XML_NS = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"
REL_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
INVALID_XML_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

def xml_safe_text(value):
    text = "" if value is None else str(value)
    text = INVALID_XML_RE.sub("�", text)
    if len(text) > MAX_EXCEL_CELL_CHARS:
        raise ValueError(f"Cellule trop longue pour Excel : {len(text):,} caractères")
    return text

def column_letter(index):
    result = ""
    while index:
        index, remainder = divmod(index - 1, 26)
        result = chr(65 + remainder) + result
    return result

def build_sheet_xml(headers, rows):
    ET.register_namespace("", XML_NS)
    worksheet = ET.Element(f"{{{XML_NS}}}worksheet")
    max_column = column_letter(len(headers))
    ET.SubElement(worksheet, f"{{{XML_NS}}}dimension", ref=f"A1:{max_column}{len(rows) + 1}")

    sheet_views = ET.SubElement(worksheet, f"{{{XML_NS}}}sheetViews")
    sheet_view = ET.SubElement(sheet_views, f"{{{XML_NS}}}sheetView", workbookViewId="0")
    ET.SubElement(sheet_view, f"{{{XML_NS}}}pane", ySplit="1", topLeftCell="A2", activePane="bottomLeft", state="frozen")

    columns = ET.SubElement(worksheet, f"{{{XML_NS}}}cols")
    ET.SubElement(columns, f"{{{XML_NS}}}col", min="1", max="1", width="18", customWidth="1")
    if len(headers) > 1:
        ET.SubElement(columns, f"{{{XML_NS}}}col", min="2", max=str(len(headers)), width="60", customWidth="1")

    sheet_data = ET.SubElement(worksheet, f"{{{XML_NS}}}sheetData")

    def add_row(excel_row, values, numeric_first=False):
        row_element = ET.SubElement(sheet_data, f"{{{XML_NS}}}row", r=str(excel_row))
        for column_index, value in enumerate(values, start=1):
            reference = f"{column_letter(column_index)}{excel_row}"
            if numeric_first and column_index == 1:
                cell = ET.SubElement(row_element, f"{{{XML_NS}}}c", r=reference)
                ET.SubElement(cell, f"{{{XML_NS}}}v").text = str(int(value))
            else:
                cell = ET.SubElement(row_element, f"{{{XML_NS}}}c", r=reference, t="inlineStr")
                inline = ET.SubElement(cell, f"{{{XML_NS}}}is")
                text_element = ET.SubElement(inline, f"{{{XML_NS}}}t")
                text_element.set("{http://www.w3.org/XML/1998/namespace}space", "preserve")
                text_element.text = xml_safe_text(value)

    add_row(1, headers)
    for excel_row, values in enumerate(rows, start=2):
        add_row(excel_row, values, numeric_first=True)

    return ET.tostring(worksheet, encoding="utf-8", xml_declaration=True)

def write_xlsx(path, headers, rows):
    sheet_xml = build_sheet_xml(headers, rows)
    content_types = b'''<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">
<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>
<Default Extension="xml" ContentType="application/xml"/>
<Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/>
<Override PartName="/xl/worksheets/sheet1.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/>
</Types>'''
    root_rels = b'''<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">
<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/>
</Relationships>'''
    workbook = b'''<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships">
<sheets><sheet name="traduction" sheetId="1" r:id="rId1"/></sheets>
</workbook>'''
    workbook_rels = b'''<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">
<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet1.xml"/>
</Relationships>'''

    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with zipfile.ZipFile(temporary_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        archive.writestr("[Content_Types].xml", content_types)
        archive.writestr("_rels/.rels", root_rels)
        archive.writestr("xl/workbook.xml", workbook)
        archive.writestr("xl/_rels/workbook.xml.rels", workbook_rels)
        archive.writestr("xl/worksheets/sheet1.xml", sheet_xml)
    temporary_path.replace(path)

    with zipfile.ZipFile(path) as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Archive XLSX corrompue ({bad_member}) : {path}")
    if path.stat().st_size > MAX_GOOGLE_DOCUMENT_BYTES:
        raise ValueError(f"{path.name} dépasse la limite Google de 10 Mo")

## 3. Construire les pilotes

Les noms techniques des colonnes seront peut-être eux aussi traduits par Google. Ce n'est pas un problème : le futur importeur utilisera leur position et l'index numérique de la première colonne.

In [ ]:
TRANSLATION_FIELDS = {
    "fr_bau": ["output_fr"],
    "bau_bau": ["instruction_fr", "input_fr", "output_fr"],
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

def export_batch(direction, records, output_path):
    fields = TRANSLATION_FIELDS[direction]
    headers = ["source_row_index_DO_NOT_TRANSLATE", *fields]
    rows = [
        [record["source_row_index"], *(record[field] for field in fields)]
        for record in records
    ]
    write_xlsx(output_path, headers, rows)
    return {
        "file": str(output_path.relative_to(PROJECT_ROOT)),
        "direction": direction,
        "rows": len(records),
        "fields_to_translate": fields,
        "first_source_row_index": records[0]["source_row_index"],
        "last_source_row_index": records[-1]["source_row_index"],
        "source_ids": [record["id"] for record in records],
        "xlsx_bytes": output_path.stat().st_size,
        "xlsx_sha256": sha256_file(output_path),
    }

pilot_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Google Translate document pilot: French to Baoulé",
    "target_language": "Baoulé",
    "target_language_code": "bau",
    "batches": [],
}

for direction, records in source_partitions.items():
    output_path = PILOT_UPLOAD_DIR / f"pilot_{direction}_0001.xlsx"
    metadata = export_batch(direction, records[:PILOT_ROWS], output_path)
    pilot_manifest["batches"].append(metadata)
    print(f"Créé : {output_path} — {metadata['rows']} lignes — {metadata['xlsx_bytes']:,} octets")

pilot_manifest_path = STEP_DIR / "pilot_manifest.json"
pilot_manifest_path.write_text(
    json.dumps(pilot_manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("Manifeste pilote :", pilot_manifest_path)

## 4. Procédure manuelle pour traduire les deux pilotes

Pour chaque fichier présent dans `data/02_google_translate/pilot_to_upload/` :

1. ouvrir <https://translate.google.com/> sur un ordinateur ;
2. sélectionner **Documents** ;
3. choisir **Français** comme langue source et **Baoulé** comme langue cible ;
4. importer le classeur, lancer la traduction, puis télécharger le résultat ;
5. ne pas modifier les lignes ni les colonnes ;
6. renommer les résultats exactement comme indiqué ci-dessous et les déposer dans `data/02_google_translate/pilot_translated/`.

Noms attendus :

- `pilot_fr_bau_0001_translated.xlsx`
- `pilot_bau_bau_0001_translated.xlsx`

Le prochain notebook vérifiera les index, le nombre de cellules, les traductions vides, les textes restés identiques et les caractères Unicode avant d'autoriser la génération complète.

## 5. Génération complète — désactivée jusqu'à validation du pilote

Cette cellule produira environ 12 fichiers par direction avec la taille actuelle du dataset. Elle ne s'exécute que si `GENERATE_FULL_BATCHES = True`.

In [ ]:
if not GENERATE_FULL_BATCHES:
    print("Génération complète ignorée : valider d'abord les deux pilotes.")
else:
    full_manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "batch_size": BATCH_SIZE,
        "target_language": "Baoulé",
        "target_language_code": "bau",
        "batches": [],
    }
    for direction, records in source_partitions.items():
        batch_count = math.ceil(len(records) / BATCH_SIZE)
        for batch_index in range(batch_count):
            start = batch_index * BATCH_SIZE
            batch_records = records[start:start + BATCH_SIZE]
            output_path = FULL_UPLOAD_DIR / f"{direction}_{batch_index + 1:04d}.xlsx"
            metadata = export_batch(direction, batch_records, output_path)
            full_manifest["batches"].append(metadata)
            print(
                f"Créé : {output_path.name} — {len(batch_records):,} lignes "
                f"— {output_path.stat().st_size / 1024:.1f} Kio"
            )

    full_manifest_path = STEP_DIR / "full_manifest.json"
    full_manifest_path.write_text(
        json.dumps(full_manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print("Manifeste complet :", full_manifest_path)

## Résultat attendu de cette étape

Les deux fichiers pilotes à envoyer à Google se trouvent dans `data/02_google_translate/pilot_to_upload/`. Ne générez pas encore les lots complets : la capacité de Google à préserver le classeur et à traduire vers le baoulé doit d'abord être confirmée sur ces 200 exemples.